# VLMEvalKit Outputs 결과 확인

In [ ]:
import sys
sys.path.append('..')

In [ ]:
import re
import pandas as pd
from pathlib import Path

OUTPUT_DIR = Path("outputs")

## 1. 전체 결과 로드

In [ ]:
score_files = sorted([f for f in OUTPUT_DIR.rglob("*_score.*") if f.suffix in ('.xlsx', '.tsv')])
all_dfs = {}
for f in score_files:
    key = str(f.relative_to(OUTPUT_DIR))
    if f.suffix == '.xlsx':
        df = pd.read_excel(f)
    else:
        df = pd.read_csv(f, sep='\t')
    all_dfs[key] = df
    print(f"{key}: {len(df)} samples, score={df['score'].mean()*100:.2f}%")

Qwen3-VL/Qwen3-VL-2B-Instruct/Qwen3-VL-2B-Instruct_MLVU_tp28000_mf256_score.xlsx: 2174 samples, score=69.09%
Qwen3-VL/Qwen3-VL-2B-Instruct/Qwen3-VL-2B-Instruct_MLVU_tp56000_mf512_score.xlsx: 2174 samples, score=69.27%
Qwen3-VL/Qwen3-VL-2B-Instruct/Qwen3-VL-2B-Instruct_Video-MME_tp14000_mf128_score.tsv: 2700 samples, score=61.19%
Qwen3-VL/Qwen3-VL-2B-Instruct/Qwen3-VL-2B-Instruct_Video-MME_tp1750_mf16_score.xlsx: 2700 samples, score=53.04%
Qwen3-VL/Qwen3-VL-2B-Instruct/Qwen3-VL-2B-Instruct_Video-MME_tp28000_mf256_score.xlsx: 2700 samples, score=62.19%
Qwen3-VL/Qwen3-VL-2B-Instruct/Qwen3-VL-2B-Instruct_Video-MME_tp3500_mf32_score.tsv: 2700 samples, score=56.56%
Qwen3-VL/Qwen3-VL-2B-Instruct/Qwen3-VL-2B-Instruct_Video-MME_tp56000_mf512_score.xlsx: 2700 samples, score=63.37%
Qwen3-VL/Qwen3-VL-2B-Instruct/Qwen3-VL-2B-Instruct_Video-MME_tp7000_mf64_score.tsv: 2700 samples, score=59.37%
Qwen3-VL/Qwen3-VL-4B-Instruct/Qwen3-VL-4B-Instruct_MLVU_tp28000_mf256_score.xlsx: 2174 samples, score=73.83

## 2. 정확도 요약

In [ ]:
summary_rows = []
for key, df in all_dfs.items():
    correct = df["score"].sum()
    total = len(df)
    acc = correct / total * 100
    summary_rows.append({"file": key, "correct": int(correct), "total": total, "accuracy": f"{acc:.2f}%"})

summary_df = pd.DataFrame(summary_rows)
summary_df

,file,correct,total,accuracy
0,Qwen3-VL/Qwen3-VL-2B-Instruct/Qwen3-VL-2B-Inst...,1502,2174,69.09%
1,Qwen3-VL/Qwen3-VL-2B-Instruct/Qwen3-VL-2B-Inst...,1506,2174,69.27%
2,Qwen3-VL/Qwen3-VL-2B-Instruct/Qwen3-VL-2B-Inst...,1652,2700,61.19%
3,Qwen3-VL/Qwen3-VL-2B-Instruct/Qwen3-VL-2B-Inst...,1432,2700,53.04%
4,Qwen3-VL/Qwen3-VL-2B-Instruct/Qwen3-VL-2B-Inst...,1679,2700,62.19%
5,Qwen3-VL/Qwen3-VL-2B-Instruct/Qwen3-VL-2B-Inst...,1527,2700,56.56%
6,Qwen3-VL/Qwen3-VL-2B-Instruct/Qwen3-VL-2B-Inst...,1711,2700,63.37%
7,Qwen3-VL/Qwen3-VL-2B-Instruct/Qwen3-VL-2B-Inst...,1603,2700,59.37%
8,Qwen3-VL/Qwen3-VL-4B-Instruct/Qwen3-VL-4B-Inst...,1605,2174,73.83%
9,Qwen3-VL/Qwen3-VL-4B-Instruct/Qwen3-VL-4B-Inst...,1857,2700,68.78%


In [ ]:
## 3. 결과 요약 테이블 (mf를 column으로)

rows = []
for f in score_files:
    rel = f.relative_to(OUTPUT_DIR)
    parts = rel.parts

    simple = "simple" if parts[0].endswith("-simple") else ""
    model = parts[1]

    m = re.search(r'_(Video-MME|MLVU)_tp(\d+)_mf(\d+)', f.stem)
    if not m:
        continue
    dataset = m.group(1)
    mf = int(m.group(3))

    key = str(rel)
    df = all_dfs[key]
    acc = df["score"].mean() * 100

    rows.append({
        "model": model,
        "dataset": dataset,
        "simple": simple,
        "mf": mf,
        "accuracy": round(acc, 2),
        "total": len(df),
    })

result_df = pd.DataFrame(rows)

# mf를 column으로 pivot
pivot = result_df.pivot_table(
    index=["model", "dataset", "simple"],
    columns="mf",
    values="accuracy",
)
pivot.columns = [f"mf={c}" for c in pivot.columns]
pivot = pivot.reset_index()

# total 정보도 추가
pivot_total = result_df.pivot_table(
    index=["model", "dataset", "simple"],
    columns="mf",
    values="total",
)
pivot_total.columns = [f"n={c}" for c in pivot_total.columns]
pivot_total = pivot_total.reset_index()

final = pivot.merge(pivot_total, on=["model", "dataset", "simple"])
acc_cols = sorted([c for c in final.columns if c.startswith("mf=")], key=lambda x: int(x.split("=")[1]))
n_cols = sorted([c for c in final.columns if c.startswith("n=")], key=lambda x: int(x.split("=")[1]))
final = final[["model", "dataset", "simple"] + acc_cols + n_cols]
final

,model,dataset,simple,mf=16,mf=32,mf=64,mf=128,mf=256,mf=512,n=16,n=32,n=64,n=128,n=256,n=512
0,Qwen3-VL-2B-Instruct,MLVU,,NaN,NaN,NaN,NaN,69.09,69.27,NaN,NaN,NaN,NaN,2174.0,2174.0
1,Qwen3-VL-2B-Instruct,Video-MME,,53.04,56.56,59.37,61.19,62.19,63.37,2700.0,2700.0,2700.0,2700.0,2700.0,2700.0
2,Qwen3-VL-4B-Instruct,MLVU,,NaN,NaN,NaN,NaN,73.83,NaN,NaN,NaN,NaN,NaN,2174.0,NaN
3,Qwen3-VL-4B-Instruct,Video-MME,,NaN,NaN,NaN,NaN,68.78,NaN,NaN,NaN,NaN,NaN,2700.0,NaN
